# VM — hq 처방을 Can에: `can_tent12i_hq` × 3 (+ 선택: `square_tent12_hq` seed 4·5), 그리고 9/7 밤 결과 회수 (GitHub에서 바로 여는 판)

**질문 하나만 답한다: "critic 타깃에서 엔트로피 보너스를 빼는 처방(β=0)이 Can에서는 해가 없는가."**
Square에서는 닫혔다(HANDOFF 20.5: `square_tent12_hq` 127k 0.49 vs baseline 0.40 vs tent12 0.37, 42k 0.33 = 판정선). Can은 원래 문제가 안 드러나는 과제(α ≤ 0.5, γ 0.99)라 이득이 아니라 **무해**를 확인하는 실험이다. 이게 있어야 "두 과제 공통 처방"이라고 쓸 수 있다.

**상호작용 가설을 명시한다**: `can_hardq`(β=0, 목표 엔트로피 0, n=3)는 baseline보다 나빴다(최저 0.10, AUC 0.30). 그러니 "β=0이 좋다"가 아니라 **"엔트로피를 붙든 상태(목표 12, 초기 α 0.3)에서만 β=0이 이득/무해"** 가 가설이다.

**사전 판정(150k, 5k 격자, n=3, `plot_results.py` 기준)**
- 무해: 평균곡선 최저 ≥ 0.405(π_dp 기준선; tent12i 0.46, 고정 0.3 0.47) **그리고** 129k ≥ 0.53(baseline n=5).
- 해로움: 최저 < 0.405(dip 복귀, hardq 방향) 또는 129k < 0.53 → "처방은 γ가 큰 과제 전용"으로 좁힌다.
- 진단: `logp_mean` ≈ −12 유지(엔트로피 12), `q_start − mc_return`이 tent12i·고정 0.3(+200 이상)보다 0에 가까워야 한다(보너스가 빠졌으니 Q_W가 실제 수익 눈금).

비교군(모두 있음): `can_tent12i`(n=3), `can_fixalpha_03`(n=5), `can_baseline`(n=5), `can_hardq`(n=3, 100k).

순서: 0 → 1(재시작) → 2 → 3 → 4 → 5 → 6 → 7 → 8 → **9(회수·상태)** → 10 → 11 → 12 keepalive → (끝) 13 zip.
Square seed 4·5를 같은 VM에서 돌리면 5b(Square 체크포인트) → 10b.

## 0. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJ = '/content/drive/MyDrive/dsrl_project'
for d in ['ckpt', 'logs', 'cfg_backup', 'dppo_log']:
    os.makedirs(f'{PROJ}/{d}', exist_ok=True)
print('project dir:', PROJ)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. condacolab (실행 뒤 런타임이 자동 재시작된다. 재시작되면 2번부터)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()   # 여기서 커널 재시작

## 2. 재시작 후: Drive 다시 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/dsrl_project'
print('ok')

## 3. 저장소 클론 (브랜치 o2o). 두 서브모듈 폴더가 비어 있으면 다음 셀로 다시 받기

In [ ]:
%%bash
git config --global url."https://github.com/".insteadOf "git@github.com:"

cd /content
rm -rf dsrl
git clone --recurse-submodules -b o2o https://github.com/msp0617/dsrl.git
cd dsrl

git log --oneline -1
echo "=== dppo ==="
ls dppo | head -3
echo "=== stable-baselines3 ==="
ls stable-baselines3 | head -3

In [ ]:
%%bash
cd /content/dsrl
git submodule sync --recursive
git submodule update --init --recursive
ls dppo | head

## 4. conda 환경 복원 (Drive 캐시 `env_cache/dsrl_env.tar.gz`, 3~5분)
캐시가 없으면 v3 노트북의 4~5절(설치, 15분)을 대신 돌리고 5b 저장 셀로 캐시를 만들어 둔다.

In [ ]:
%%bash
# 복원: 새 VM에서 4~5번 대신. 1번(condacolab), 2번(Drive), 3번(클론) 뒤에 실행.
set -e
CACHE=/content/drive/MyDrive/dsrl_project/env_cache
mkdir -p /usr/local/envs
cd /usr/local/envs
rm -rf dsrl
tar -xzf $CACHE/dsrl_env.tar.gz
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python - <<'PY'
import torch, robomimic, robosuite, mujoco, stable_baselines3
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| cap", torch.cuda.get_device_capability(0))
x = torch.randn(256, 256, device="cuda"); print("matmul ok", (x @ x).sum().item() != 0)
print("robomimic", robomimic.__version__, "robosuite", robosuite.__version__, "mujoco", mujoco.__version__)
PY
echo "restored; continue from section 6"

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python /usr/local/envs/dsrl/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py

## 5. π_dp 체크포인트 (Can). Drive `dppo_log/`에서 config가 기대하는 상대경로로 복사

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
mkdir -p /content/dsrl/dppo/log

if [ -z "$(ls -A $PROJ/dppo_log 2>/dev/null)" ]; then
  echo "--- 첫 다운로드 ---"
  cd /content/dsrl/dppo/log
  gdown --folder https://drive.google.com/drive/folders/1kzC49RRFOE7aTnJh_7OvJ1K5XaDmtuh1
  cp -r /content/dsrl/dppo/log/. $PROJ/dppo_log/
else
  echo "--- Drive에서 복사 ---"
  cp -r $PROJ/dppo_log/. /content/dsrl/dppo/log/
fi
find /content/dsrl/dppo/log -maxdepth 3 | head -40

In [ ]:
%%bash
set -e

RUNTIME=/content/dsrl/dppo/log
DRIVE=/content/drive/MyDrive/dsrl_project/dppo_log

CKPT_REL=robomimic-pretrain/can/can_pre_diffusion_mlp_ta4_td20/2024-06-28_13-29-54/checkpoint/state_5000.pt
NORM_REL=robomimic/can/normalization.npz

mkdir -p "$RUNTIME" "$DRIVE"

CKPT_SRC=$(find "$RUNTIME" "$DRIVE" \
  -type f -path "*/$CKPT_REL" -print -quit 2>/dev/null || true)

NORM_SRC=$(find "$RUNTIME" "$DRIVE" \
  -type f -path "*/$NORM_REL" -print -quit 2>/dev/null || true)

echo "checkpoint: ${CKPT_SRC:-NOT_FOUND}"
echo "normalization: ${NORM_SRC:-NOT_FOUND}"

if [[ -z "$CKPT_SRC" || -z "$NORM_SRC" ]]; then
  echo "기존 다운로드에서 파일을 찾지 못했습니다."
  exit 2
fi

CKPT_DST="$RUNTIME/$CKPT_REL"
NORM_DST="$RUNTIME/$NORM_REL"

mkdir -p "$(dirname "$CKPT_DST")" "$(dirname "$NORM_DST")"

[[ "$CKPT_SRC" == "$CKPT_DST" ]] || cp -f "$CKPT_SRC" "$CKPT_DST"
[[ "$NORM_SRC" == "$NORM_DST" ]] || cp -f "$NORM_SRC" "$NORM_DST"

# 다음 세션을 위해 Drive에도 정확한 구조로 저장
mkdir -p "$DRIVE/$(dirname "$CKPT_REL")"
mkdir -p "$DRIVE/$(dirname "$NORM_REL")"
cp -f "$CKPT_DST" "$DRIVE/$CKPT_REL"
cp -f "$NORM_DST" "$DRIVE/$NORM_REL"

echo "=== READY ==="
ls -lh "$CKPT_DST" "$NORM_DST"

## 6. 환경 변수

In [ ]:
%%bash
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS

cat /content/env.sh

## 7. 실행 헬퍼 `run_bash`

In [ ]:
# Colab의 %%bash 는 명령이 끝나야 출력을 보여준다. 긴 학습은 이 헬퍼로 돌려서
# 한 줄씩 바로 보이게 하고, 같은 내용을 Drive의 로그 파일에도 남긴다.
import subprocess, sys

NOISE = ("Gym has been unmaintained", "Please upgrade to Gymnasium", "See the migration guide")

def run_bash(script, log_path=None):
    prefix = (
        "source /usr/local/etc/profile.d/conda.sh && conda activate dsrl\n"
        "source /content/env.sh\n"
        "cd /content/dsrl\n"
        "export HYDRA_FULL_ERROR=1 PYTHONUNBUFFERED=1\n"
    )
    log = open(log_path, "a") if log_path else None
    p = subprocess.Popen(["bash", "-c", prefix + script], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if log:
            log.write(line); log.flush()
        if not line.startswith(NOISE):
            print(line, end="", flush=True)
    p.wait()
    if log:
        log.close()
    print(f"\n[exit {p.returncode}]")
    return p.returncode

PROJ = "/content/drive/MyDrive/dsrl_project"
print("run_bash ready")

## 8. site-packages 패치

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python /content/dsrl/colab/patch_env.py

## 5b. (Square seed 4·5를 돌릴 때만) Square π_dp 체크포인트

In [ ]:
%%bash
# Square의 π_dp와 normalization을 config가 기대하는 상대경로에 놓는다 (6번 Can 셀과 같은 방식).
# Drive의 dppo_log 안 어디에 있든 찾아서 복사한다. Square run을 띄울 때만 필요.
set -e
RUNTIME=/content/dsrl/dppo/log
DRIVE=/content/drive/MyDrive/dsrl_project/dppo_log
CKPT_REL=robomimic-pretrain/square/square_pre_diffusion_mlp_ta4_td100_ddim-100steps/2025-04-11_19-13-26_44/checkpoint/state_3000.pt
NORM_REL=robomimic/square/normalization.npz
CKPT_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$CKPT_REL" -print -quit 2>/dev/null || true)
NORM_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$NORM_REL" -print -quit 2>/dev/null || true)
echo "checkpoint:    ${CKPT_SRC:-NOT_FOUND}"
echo "normalization: ${NORM_SRC:-NOT_FOUND}"
[[ -n "$CKPT_SRC" && -n "$NORM_SRC" ]] || { echo "square files missing in $DRIVE"; exit 2; }
mkdir -p "$RUNTIME/$(dirname $CKPT_REL)" "$RUNTIME/$(dirname $NORM_REL)"
[[ "$CKPT_SRC" == "$RUNTIME/$CKPT_REL" ]] || cp -f "$CKPT_SRC" "$RUNTIME/$CKPT_REL"
[[ "$NORM_SRC" == "$RUNTIME/$NORM_REL" ]] || cp -f "$NORM_SRC" "$RUNTIME/$NORM_REL"
echo "=== SQUARE READY ==="; ls -lh "$RUNTIME/$CKPT_REL" "$RUNTIME/$NORM_REL"


## 9. 결과 회수 — hq 7 run + 9/7 밤 24 run(`can_{td,cql,calql}`, `can_calql_{t12i,prefill}`, `square_{td,cql,calql}`)의 상태
포스터의 `[tonight]` 칸이 이 run들이다. 돌리기 전에 **뭐가 이미 있는지**부터 본다. `[done]`이면 완료, `[eval]`이면 마지막 평가까지 진행, 없으면 시작 안 됨/죽음(`.out`이 없으면 "없음").

In [ ]:
%%bash
PROJ=/content/drive/MyDrive/dsrl_project
RUNS="square_tent12_hq_s1 square_tent12_hq_s2 square_tent12_hq_s3 square_tent12_s4 square_tent12_s5 square_baseline_s4 square_baseline_s5"
for M in td cql calql; do for S in 1 2 3; do RUNS="$RUNS can_${M}_s$S square_${M}_s$S"; done; done
for S in 1 2 3; do RUNS="$RUNS can_calql_t12i_s$S can_calql_prefill_s$S"; done
for E in $RUNS; do
  if [ -f $PROJ/logs/$E.out ]; then
    printf "%-24s %s\n" "$E:" "$(grep '\[done\]\|\[eval\]' $PROJ/logs/$E.out | tail -n 1 | cut -c1-90)"
  else
    printf "%-24s (없음)\n" "$E:"
  fi
done
echo; ls $PROJ/logs/pretrain/*.pt 2>/dev/null | xargs -n1 basename | tr '\n' ' '; echo

## 10. 온라인 — `can_tent12i_hq_s{1,2,3}`, 150k, 5k 격자 (RAM ≈ 51 GB, G4에서 약 5~6시간)
tent12i(`train.ent_coef=auto_0.3 train.target_ent=12`) + `train.critic_entropy_scale=0.0`. 사전학습·리플레이 없음(`variant=baseline`, `offline_mix.mode=none`, `load_offline_data=False` = 기본값). 띄운 뒤 12번 keepalive.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && source /content/env.sh && cd /content/dsrl
git pull origin o2o | tail -n 1
SEEDS="1 2 3"
PROJ=/content/drive/MyDrive/dsrl_project
CFG="--config-path=cfg/robomimic --config-name=dsrl_can.yaml"
COMMON="log_dir=$PROJ/logs train.total_env_steps=150000 offline_mix.mode=none load_offline_data=False"
launch () { EXP=$1; shift; nohup python train_dsrl.py $CFG exp_id=$EXP "$@" $COMMON > $PROJ/logs/$EXP.out 2>&1 & echo "started $EXP (pid $!)"; }
for S in $SEEDS; do
  launch can_tent12i_hq_s$S seed=$S variant=baseline train.ent_coef=auto_0.3 train.target_ent=12 train.critic_entropy_scale=0.0
done
sleep 120; free -g | head -2

## 10b. (선택) `square_tent12_hq_s{4,5}` — Square hq를 baseline·tent12와 같은 n=5로
Square 127k의 seed SD ≈ 0.13이라 n=5로도 "시사적"을 못 벗어난다(0.1 차이를 80% power로 보려면 arm당 ~25). 그래도 비교군과 n을 맞추는 값어치는 있다(2 run, 150k, Square는 RAM 더 씀). 5b를 먼저 실행.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && source /content/env.sh && cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
CFG="--config-path=cfg/robomimic --config-name=dsrl_square.yaml"
COMMON="log_dir=$PROJ/logs train.total_env_steps=150000 offline_mix.mode=none load_offline_data=False"
launch () { EXP=$1; shift; nohup python train_dsrl.py $CFG exp_id=$EXP "$@" $COMMON > $PROJ/logs/$EXP.out 2>&1 & echo "started $EXP (pid $!)"; }
for S in 4 5; do
  launch square_tent12_hq_s$S seed=$S variant=baseline train.target_ent=12 train.critic_entropy_scale=0.0
done
sleep 120; free -g | head -2

## 11. 확인 (3~5분 뒤). 기대: `[eval] env_steps=0`이 찍히고, `.out`에 `critic_entropy_scale`이 0.0으로 fingerprint에 들어감. 20~30분 뒤 `train_log.csv`의 `ent_coef`가 0.3 근처에서 움직이고 `logp_mean`이 −12 쪽으로 가는지

In [ ]:
%%bash
PROJ=/content/drive/MyDrive/dsrl_project
echo "processes: $(ps aux | grep -c '[t]rain_dsrl.py') (run 수 x 2)"
for F in $(ls -t $PROJ/logs/can_tent12i_hq_s*.out $PROJ/logs/square_tent12_hq_s[45].out 2>/dev/null); do
  echo "== $(basename $F .out): $(grep '\[eval\]\|Error\|Traceback' $F | tail -n 1 | cut -c1-110)"
done
# 마지막 train_log 행의 env_steps / ent_coef / logp_mean / qw_mean (열 이름으로 찾음)
for T in $(ls $PROJ/logs/can_tent12i_hq_s*/train_log.csv $PROJ/logs/square_tent12_hq_s[45]/train_log.csv 2>/dev/null); do
  echo "$(basename $(dirname $T)): $(awk -F, 'NR==1{for(i=1;i<=NF;i++)c[$i]=i} END{printf "env_steps=%s ent_coef=%s logp_mean=%s qw_mean=%s", $c["env_steps"], $c["ent_coef"], $c["logp_mean"], $c["qw_mean"]}' $T)"
done
free -g | head -2

## 12. keepalive (사전학습·온라인 공용). 10분마다 한 줄, 프로세스가 없으면 런타임을 스스로 반납한다
다른 셀을 돌려야 하면 이 셀 정지 → 셀 실행 → 다시 실행. 반납 없이 붙들어 두려면 `runtime.unassign()` 줄을 지운다.

In [ ]:
import subprocess, time
PROJ = '/content/drive/MyDrive/dsrl_project'
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
while True:
    running = sh("ps aux | grep '[o]ffline_pretrain.py\\|[t]rain_dsrl.py' | grep -o 'exp_id=[a-z_0-9]*\\|pretrain.method=[a-z]*\\|seed=[0-9]*' | tr '\\n' ' '")
    ram = sh("free -g | awk 'NR==2{print $3\"/\"$2}'")
    prog = sh("for f in $(ls -t %s/logs/can_tent12i_hq_s*.out %s/logs/square_tent12_hq_s*.out 2>/dev/null | head -n 12); do "
              "n=$(basename $f .out); l=$(grep '^\\[cql\\]\\|^\\[calql\\]\\|^\\[distill\\]\\|\\[eval\\]\\|\\[done\\]' $f | tail -n 1 | cut -c1-70); "
              "echo -n \"$n: $l | \"; done" % (PROJ, PROJ))
    print(time.strftime('%H:%M'), 'ram', ram, '|', running or '(none running)', '|', prog, flush=True)
    if not running:
        print('all done -> unassigning runtime', flush=True)
        from google.colab import runtime
        runtime.unassign()
        break
    time.sleep(600)

## 13. 결과 zip (CPU 런타임에서도 됨: 0번 Drive 마운트 → 이 셀). 로컬에서 `~/Downloads/logs/`에 풀고
`.venv/bin/python scripts/plot_results.py --logs ~/Downloads/logs --out ~/Downloads/dsrl_figs_hq --axes "hq=baseline,tent12i,fixalpha_03,hardq,tent12i_hq;square_hq=square_baseline,square_tent12,square_tent12_hq;critic=baseline,iql,td,cql,calql,warmupc"`

In [ ]:
%%bash
cd /content/drive/MyDrive/dsrl_project
rm -f csv_bundle.zip
zip -qr csv_bundle.zip logs -i "logs/*/eval_log.csv" "logs/*/train_log.csv" "logs/*.csv" "logs/pretrain/*_log.csv"
ls -lh csv_bundle.zip